## Model Refinement

In this notebook, we improve the regression model by addressing heteroscedasticity observed in the residual analysis.

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/insurance_clean.csv")
df["smoker_encoded"] = df["smoker"].map({"no": 0, "yes": 1})
df["bmi_smoker_interaction"] = df["bmi"] * df["smoker_encoded"]
df["log_charges"] = np.log(df["charges"])

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Prepare features and log target
x = df[["age", "bmi", "children", "smoker_encoded", "bmi_smoker_interaction"]] # x = predictors (independent variables))
y_log = df["log_charges"] # y_log = log-transformed target variable
# We use log because original charges had heteroscedasticity

# We split data into training and testing sets (80% train, 20% test)
# Train data (model learns patterns)
# Test data (model performance evaluation)
x_train, x_test, y_train, y_test = train_test_split(x, y_log, test_size=0.2, random_state=42)

# Train linear regression model on log-transformed target
# Model now predicts log(charges) instead of charges
model_log = LinearRegression()
model_log.fit(x_train, y_train)

# Evaluate R² on log scale
# This tells how much variance in log(charges) is explained by the model
r2_log = model_log.score(x_test, y_test)
r2_log

0.827621679904847

In [15]:
# Convert predictions back to original scale
# We predict log(charges) and then apply exp to get back to charges
y_log_pred = model_log.predict(x_test)

#True original charges for comparison
y_pred_charges = np.exp(y_log_pred)
y_true_charges = df.loc[y_test.index, "charges"]

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Evaluate error in real monetary units
mae = mean_absolute_error(y_true_charges, y_pred_charges) # MAE = average absolute difference between true and predicted charges
rmse = np.sqrt(mean_squared_error(y_true_charges, y_pred_charges)) # RMSE = penalizes larger errors more strongly than MAE

mae, rmse

(4144.859566946494, np.float64(8464.668354424675))

### Model Refinament - Final Evaluation

Although heteroscedasticity was observed in the residual analysis of the baseline model, applying a logarytmic transformation to the target variable did not improve predictive performance when evaluated in the original scale of insurance charges.

While the log-transformed model achieved a resonable R² (~0.83), its MAE and RMSE were higher after back-transformation compared to the interaction model.

The interaction model (including  BMI x Smoking) remains the best-performing specification:
- It captures the strong conditional effect between BMI and smoking.
- it achieves lower prediction errors in real monetary units.
- It maintains interpretability in the original cost scale.

Therefore, the interaction model without log transformation is selected as the final model.

## Business Interpretation

The analysis shows that smoking status is the dominant driver of insurance costs.

BMI significantly increases insurance charges among smokers, confirming a strong conditional effect. This suggests that lifestyle risk factors compound medical costs exposure.

Age contributes steadily to cost increases, while the number of children has only a marginal effect.

From a risk perspective, insurance pricing models should account for interaction effects between behavioral and health-related variables rather than treating predictors independently.